# Tablas — archivos limpios

Cuaderno **visual**: muestra cómo quedaron los datos después de la limpieza realizada en `Lectura.ipynb`.

- **Solo LEE** la carpeta `data/limpio/`. **No modifica ni limpia nada.**
- Para cada archivo limpio se muestra: filas, columnas y una **muestra de 3 filas** como tabla.
- El resumen global incluye filas, columnas, nulos y duplicados exactos por archivo.

## 1. Configuración y archivos limpios disponibles

Se listan automáticamente los archivos con sufijo `_limpio` dentro de `data/limpio/`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display as _display
except Exception:
    def _display(obj, *args, **kwargs):
        print(obj.to_string() if hasattr(obj, "to_string") else obj)

# Raiz del proyecto = carpeta que contiene data/ y script/.
# Si el cuaderno se abre desde la carpeta script/, subimos un nivel.
RAIZ = Path.cwd()
if not (RAIZ / "data").exists() and (RAIZ.parent / "data").exists():
    RAIZ = RAIZ.parent
LIMPIO = RAIZ / "data" / "limpio"

print("Raíz del proyecto:", RAIZ)
print("Carpeta de archivos limpios:", LIMPIO)

# Solo archivos que YA son versiones limpias (_limpio.csv / _limpio.parquet)
archivos_limpios = sorted(LIMPIO.rglob("*"))
archivos_limpios = [
    p
    for p in archivos_limpios
    if p.is_file()
    and p.suffix.lower() in (".csv", ".parquet")
    and "_limpio" in p.name
]
print(f"Archivos limpios encontrados: {len(archivos_limpios)}")


## 2. Resumen global

Tabla con filas, columnas, nulos y duplicados exactos de cada archivo limpio.

In [ ]:
filas_resumen = []
for p in archivos_limpios:
    if p.suffix.lower() == ".csv":
        df = pd.read_csv(p, dtype=str)
    else:
        df = pd.read_parquet(p)
    filas_resumen.append(
        {
            "archivo": str(p.relative_to(LIMPIO)).replace("\\", "/"),
            "filas": len(df),
            "columnas": df.shape[1],
            "nulos": int(df.isna().sum().sum()),
            "duplicados_exactos": int(df.duplicated().sum()),
        }
    )

df_resumen = pd.DataFrame(filas_resumen).sort_values("archivo")
_display(df_resumen)


## 3. Muestra de 3 filas por archivo limpio

Por cada archivo limpio se muestra una **tabla con las primeras 3 filas** (con todas sus columnas) para revisar visualmente cómo quedaron los datos.

In [ ]:
for p in archivos_limpios:
    if p.suffix.lower() == ".csv":
        df = pd.read_csv(p, dtype=str)
    else:
        df = pd.read_parquet(p)
    ruta_rel = str(p.relative_to(LIMPIO)).replace("\\", "/")
    print("=" * 78)
    print(f"{ruta_rel} | {len(df):,} filas | {df.shape[1]} columnas")
    print("-" * 78)
    _display(df.head(3))
    print()


## 4. Próximos pasos

- La **limpieza** está documentada y es reproducible en `Lectura.ipynb`.
- Este cuaderno solo **visualiza** los archivos limpios.
- La siguiente etapa (fuera de estos cuadernos) será el **cruce** `JobHop.matched_code` ↔ `occupations.code`.